# GraphRAG

## Import packages

In [42]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import nltk
import string
import numpy as np
import json
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from getpass import getpass
from unidecode import unidecode
from rouge_score import rouge_scorer
from sklearn.metrics.pairwise import cosine_similarity

from langchain_community.embeddings import OllamaEmbeddings
from langchain.embeddings.cache import CacheBackedEmbeddings
from langchain.storage import LocalFileStore

from neurorag.neurorag import NeuroRAG

## Disable warnings

In [43]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook:
1. MISTRAL_API_KEY
2. OPENAI_API_KEY
3. OPENAI_PROXY
4. TAVILY_API_KEY
5. ENTREZ_EMAIL

## Import packages

In [44]:
env_variables = [
  'MISTRAL_API_KEY',
  'OPENAI_API_KEY',
  'OPENAI_PROXY',
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Setup MEDIQA tests

In [45]:
with open('cache.json', 'r') as file:
    cache = json.load(file)

In [46]:
lemmatizer = nltk.stem.WordNetLemmatizer()

def preprocess(corpus: str) -> str:
  corpus = corpus.lower()
  stopset = nltk.corpus.stopwords.words('english') + nltk.corpus.stopwords.words('russian') + list(string.punctuation)
  tokens = nltk.word_tokenize(corpus)
  tokens = [t for t in tokens if t not in stopset]
  tokens = [lemmatizer.lemmatize(t) for t in tokens]
  corpus = ' '.join(tokens)
  corpus = unidecode(corpus)
  return corpus

In [47]:
embeddings = OllamaEmbeddings(model='llama3.1')
store = LocalFileStore("./.embeddings_cache")

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
  embeddings,
  store,
  namespace=embeddings.model,
)

In [48]:
def embeddings_cosine_sim_metric(expected_answers: list[str], predicted_answers: list[str]) -> float:
  results = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    expected_embedding = np.array(cached_embeddings.embed_query(expected_answer))
    predicted_embedding = np.array(cached_embeddings.embed_query(predicted_answer))

    sim = cosine_similarity(
      expected_embedding.reshape(1, -1),
      predicted_embedding.reshape(1, -1),
    )[0][0]

    results.append(sim)

  return np.mean(results)

In [49]:
smoothie_f = nltk.translate.bleu_score.SmoothingFunction().method4

def bleu_metric(expected_answers, predicted_answers):
  scores = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    predicted_tokens = nltk.word_tokenize(predicted_answer)
    expected_tokens = [nltk.word_tokenize(expected_answer)]

    score = nltk.translate.bleu_score.sentence_bleu(
      expected_tokens,
      predicted_tokens,
      smoothing_function=smoothie_f,
    )

    scores.append(score)

  return np.mean(scores)

In [50]:
rogue_l_scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def rogue_l_metric(expected_answers, predicted_answers):
  scores = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    result = rogue_l_scorer.score(expected_answer, predicted_answer)

    scores.append(result['rougeL'])

  return np.mean(scores)

In [51]:
rogue_1_scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)

def rogue_1_metric(expected_answers, predicted_answers):
  scores = []

  for expected_answer, predicted_answer in zip(expected_answers, predicted_answers):
    expected_answer = preprocess(expected_answer)
    predicted_answer = preprocess(predicted_answer)

    result = rogue_1_scorer.score(expected_answer, predicted_answer)

    scores.append(result['rouge1'])

  return np.mean(scores)

In [52]:
def eval_rag(app) -> float:
  dataset_df = pd.read_csv('../datasets/mediqa.csv')
  expected_answers = dataset_df['answer']
  predicted_answers = []

  for index, row in tqdm(list(dataset_df.iterrows()), desc='Questions'):
    question = row['question']

    if not 'mediqa' in cache:
      cache['mediqa'] = {}

    if question in cache['mediqa']:
      llm_answer = cache['mediqa'][question]
    else:
      response = app.invoke(question)
      llm_answer = response['generation']
      cache['mediqa'][question] = llm_answer
      with open('cache.json', 'w') as file:
        json.dump(cache, file)

    predicted_answers.append(llm_answer)

  cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
  bleu_score = bleu_metric(expected_answers, predicted_answers)
  rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
  rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)

  return cos_score, bleu_score, rogue_1_score, rogue_l_score

## Build model

In [ ]:
app = NeuroRAG(model='llama3.1', debug=False)
app.compile()

## Evaluate RAG

In [54]:
eval_rag(app)

Questions: 100%|██████████| 19/19 [00:00<00:00, 66465.20it/s]


(0.6340056405720644,
 0.0165860674773924,
 0.2737505360972866,
 0.17437035043593946)